In [17]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import plotly.express as px

In [24]:
raw = pd.read_excel(r"Signet Volume.xlsx", engine="openpyxl")

df = raw.copy()
df['Month'] = pd.to_datetime(df['Month'])
df = df.sort_values(['Serial Number', 'Month'])

df = df.groupby(['Serial Number', 'Month'], as_index=False)['Month Volume'].sum()

In [ ]:
# drop Sept 2025 -- its not all the values
df = df[df['Month']!='2025-09-01 00:00:00']

drop_sn = df[df['Month Volume']<0]['Serial Number'].unique().tolist()
df = df[~df['Serial Number'].isin(drop_sn)]

# select most current serial numbers --  keeping these
current_sn = df[df['Month']=='2025-08-01 00:00:00']['Serial Number'].unique().tolist()
df = df[df['Serial Number'].isin(current_sn)]

# removing serial numbers with no current activity
no_current_activity = df[(df['Month']=='2025-08-01 00:00:00')&(df['Month Volume']==0)]['Serial Number'].unique().tolist()
df = df[~df['Serial Number'].isin(no_current_activity)]

# Making sure that Month is first day of month (Timestamp), not a Period
df['Month'] = pd.to_datetime(df['Month']).dt.to_period('M').dt.to_timestamp()  # start of month

In [41]:
# -----------------------
# 1) Prep & feature build
# -----------------------
def ensure_monthly_grid(df):
    # df columns: Serial Number, Month, Month Volume
    df = df.copy()
    df['Month'] = pd.to_datetime(df['Month']).dt.to_period('M').dt.to_timestamp()  # month start
    df = df.sort_values(['Serial Number', 'Month'])

    out = []
    for sid, g in df.groupby('Serial Number', sort=False):
        # Month is already month-start; build a complete MS grid
        first = g['Month'].min()
        last  = g['Month'].max()
        idx = pd.date_range(start=first, end=last, freq='MS')

        gg = g.set_index('Month').reindex(idx)
        gg['Serial Number'] = sid
        gg.index.name = 'Month'
        out.append(gg.reset_index())

    full = pd.concat(out, ignore_index=True)
    return full


def add_time_features(df):
    df = df.copy()
    df['month'] = df['Month'].dt.month.astype(np.int8)
    df['year']  = df['Month'].dt.year.astype(np.int16)
    # Seasonality (12-month cycle)
    df['sin12'] = np.sin(2*np.pi*df['month']/12).astype(np.float32)
    df['cos12'] = np.cos(2*np.pi*df['month']/12).astype(np.float32)
    return df

def add_lag_features(df, lags=(1,2,12), roll_windows=(3,6,12)):
    df = df.sort_values(['Serial Number','Month']).copy()
    g = df.groupby('Serial Number', group_keys=False)
    for L in lags:
        df[f'lag_{L}'] = g['Month Volume'].shift(L)
    for W in roll_windows:
        # Shift by 1 so the window ends at t-1 to avoid leakage
        df[f'roll_mean_{W}'] = g['Month Volume'].shift(1).rolling(W).mean()
        df[f'roll_std_{W}']  = g['Month Volume'].shift(1).rolling(W).std()
    return df

def build_training_frame(raw_df):
    df = ensure_monthly_grid(raw_df)
    df = add_time_features(df)
    df = add_lag_features(df)
    return df

# -----------------------
# 2) Train / validate split
# -----------------------
def time_split(df, cutoff):
    """
    cutoff: a Timestamp (inclusive) for train end.
    Train: Month <= cutoff
    Valid: next H months after cutoff (we'll filter to rows with all features available)
    """
    train = df[df['Month'] <= cutoff].copy()
    valid = df[df['Month'] >  cutoff].copy()
    return train, valid

# -----------------------
# 3) LightGBM training
# -----------------------
def train_lgbm(train_df, target_col='Month Volume', cat_cols=('Serial Number',)):
    features = [c for c in train_df.columns if c not in ['Month', target_col]]

    # Keep only rows where lag/rolling features are available
    need = [c for c in features if c.startswith('lag_') or c.startswith('roll_')]
    X_train = train_df.dropna(subset=need).copy()
    y_train = X_train[target_col].astype(float)
    X_train = X_train[features]

    # Categorical handling: set dtype to 'category'
    for c in cat_cols:
        if c in X_train.columns:
            X_train[c] = X_train[c].astype('category')

    params = dict(
        objective='regression',
        metric='mae',
        learning_rate=0.05,
        num_leaves=63,
        feature_fraction=0.9,
        bagging_fraction=0.9,
        bagging_freq=1,
        min_data_in_leaf=50,
        n_estimators=2000,
        # IMPORTANT: use 'verbosity' (not 'verbose') to silence logs
        verbosity=-1,
        # You can add random_state for reproducibility if you want:
        # random_state=42,
    )

    model = lgb.LGBMRegressor(**params)

    # Remove 'verbose' kwarg from fit; control logging with callbacks or verbosity
    model.fit(
        X_train, y_train,
        categorical_feature=[c for c in cat_cols if c in X_train.columns],
        eval_set=[(X_train, y_train)],
        eval_metric='mae',
        callbacks=[lgb.log_evaluation(period=0)]  # silence training logs
    )

    return model, features


# -----------------------
# 4) Recursive multi-step forecast (H = periods)
# -----------------------
def forecast_recursive(df_hist, model, features, periods=6, cat_cols=('Serial Number',)):
    """
    df_hist: full dataframe with history up to the latest month per serial (may include NaNs).
    Returns a DataFrame with future rows (one per month & serial) and 'Month Volume' predictions.
    """
    df = df_hist.copy()
    last_month = df['Month'].max()
    future_months = pd.date_range(last_month + pd.offsets.MonthBegin(1), periods=periods, freq='MS')

    future_rows = []
    # We’ll forecast per serial to keep lags consistent
    for sid, g in df.groupby('Serial Number'):
        g = g.sort_values('Month').copy()
        hist = g.copy()

        for m in future_months:
            # Create a new row for month m
            row = {
                'Serial Number': sid,
                'Month': m,
            }
            # Append placeholder to compute features
            hist = pd.concat([hist, pd.DataFrame([row])], ignore_index=True)

            # Recompute minimal features for the tail (efficient enough for moderate N)
            hist[['month','year','sin12','cos12']] = add_time_features(hist)[['month','year','sin12','cos12']]

            # Keep only last ~24 rows for efficiency when computing lags
            tail = hist.sort_values('Month').groupby('Serial Number').tail(36).copy()
            tail = add_lag_features(tail)

            # Extract the just-created row with features
            x = tail[tail['Month'] == m].copy()

            # In case of very short history, fall back by filling NA lags with last observed or zeros
            lag_cols = [c for c in x.columns if c.startswith('lag_') or c.startswith('roll_')]
            if x[lag_cols].isnull().any(axis=None):
                # light fallback: fill missing lags with last observed value or 0
                last_val = hist['Month Volume'].dropna().iloc[-1] if hist['Month Volume'].notna().any() else 0.0
                x[lag_cols] = x[lag_cols].fillna(last_val)

            # Model expects the same feature set
            X_pred = x[features].copy()
            for c in cat_cols:
                if c in X_pred.columns:
                    X_pred[c] = X_pred[c].astype('category')

            yhat = float(model.predict(X_pred)[0])

            # Write back the prediction to hist to allow next step’s lags
            hist.loc[hist['Month'] == m, 'Month Volume'] = max(0.0, yhat)  # non-negativity

            future_rows.append({
                'Serial Number': sid,
                'Month': m,
                'Month Volume': max(0.0, yhat)
            })

    return pd.DataFrame(future_rows)

# -----------------------
# 5) Putting it together
# -----------------------
def lgbm_global_forecast(raw_df, periods=6, cutoff=None):
    """
    raw_df columns: Serial Number, Month, Month Volume
    cutoff: timestamp to end training. If None, uses last available month - periods
    """
    df = build_training_frame(raw_df)

    if cutoff is None:
        # Train up to the last available month (use all), then forecast periods ahead
        cutoff = df['Month'].max()

    train_df, _ = time_split(df, cutoff=cutoff)

    model, features = train_lgbm(train_df, target_col='Month Volume', cat_cols=('Serial Number',))

    # Forecast periods beyond cutoff using the full history up to cutoff
    df_hist = df[df['Month'] <= cutoff].copy()
    future_df = forecast_recursive(df_hist, model, features, periods=periods, cat_cols=('Serial Number',))

    # Output
    # History (up to cutoff) + Forecast (next periods)
    history_df = df_hist[['Serial Number','Month','Month Volume']].rename(columns={'Month Volume':'Volume'})
    history_df['Type'] = 'History'

    forecast_df = future_df.rename(columns={'Month Volume':'Volume'})
    forecast_df['Type'] = 'Forecast'

    final_df = pd.concat([history_df, forecast_df], ignore_index=True)
    return final_df, model, features


In [42]:
final_df, model, features = lgbm_global_forecast(df, periods=6)

In [43]:
final_df

,Serial Number,Month,Volume,Type
0,AAJN017002864,2024-08-01,0.000000,History
1,AAJN017002864,2024-09-01,0.000000,History
2,AAJN017002864,2024-10-01,0.000000,History
3,AAJN017002864,2024-11-01,0.000000,History
4,AAJN017002864,2024-12-01,0.000000,History
...,...,...,...,...
96921,VNB3Y17391,2025-10-01,457.371576,Forecast
96922,VNB3Y17391,2025-11-01,519.769535,Forecast
96923,VNB3Y17391,2025-12-01,733.579249,Forecast
96924,VNB3Y17391,2026-01-01,711.278705,Forecast


In [55]:
sn_value = 'U64221F2N895022'

fig = px.line(final_df[final_df['Serial Number']==f'{sn_value}'], 
              x = 'Month',
              y = 'Volume',
              color = 'Type',
              markers = 'circle',
              title = f'{sn_value}',
              template = 'plotly_dark')

fig2 = px.scatter(final_df[(final_df['Serial Number']==f'{sn_value}')&(final_df['Month'].astype(str).str.contains('-12-'))], 
              x = 'Month',
              y = 'Volume',
            #   color = 'Type',
             color_discrete_sequence=px.colors.qualitative.Set2,
              title = f'{sn_value}',
              template = 'plotly_dark')

fig.add_trace(fig2.data[0])

fig.show()